# 1kg_eur — accumulate binned cross-products, by pair class

Runs `grm_class_tool accumulate` over every GRM shard and `merge` over the
per-shard accumulators, producing one `*_merged.full.tsv` per
`(phenotype, transform, covariate_set)` combo for `09_estimators.ipynb`.

Two things differ from the previous version of this notebook, and both matter:

**1. One pass per shard, not one pass per combo.** `accumulate` takes
`--pheno-list`, which accumulates K phenotypes in a single traversal of the
shard: the 4-byte read, the binary search over bins and the pair-class cursor
are all shared, and only the multiply and the accumulator update are
per-phenotype. Previously this notebook launched one `accumulate` per
`(combo, shard)`, so the ~100 GiB of shard bytes was re-read once per combo —
at 288 combos that is ~28 TiB of reads to compute something that needs ~100 GiB.
Batching makes the shard traversal a fixed cost.

**2. Pairs are classified, not pooled.** `--pair-classes` takes
`deg1_classified.tsv` from `07_relatedness_screen.ipynb` and routes each listed
pair into a per-class copy of the bin array. PO and FS pairs have the same
expected `a_ij ~= 0.5` but differ in dominance (`d_ij` = 0 for PO, 0.25 for FS)
and in shared environment, so pooling them biases the FS bin and discards a real
contrast. Nothing is excluded: `merge` synthesises a `pooled` class that
reproduces exactly what an unclassed run would have given, so the PO mean
cross-product is a result in its own right rather than something dropped.

**Machine**: `n1-highcpu-16` (16 vCPU / ~14.4 GB). `accumulate` never touches
the genotype panel — that was plink's fixed cost, not this tool's — so memory is
driven only by the phenotype matrix and the accumulators, both sized below.
`N_CONCURRENT = 8`, half the vCPUs: n1 is 2-way SMT and assuming logical cores
are free parallelism has burned this pipeline before.

**Prerequisites:**
- `06_grm_shards.ipynb` — shards in `03_grm/shards/`
- `05_phenotype_residualize.ipynb` — `.pheno` files in `05_phenotypes/residualized/`
- `07_relatedness_screen.ipynb` — `deg1_classified.tsv`

## Config

In [ ]:
import glob
import hashlib
import os
import re
import subprocess
from datetime import datetime, timezone

import pandas as pd

# The checkout has appeared under several spellings on this VM, so resolve it
# rather than hardcode one -- a wrong guess otherwise fails obscurely inside
# `make`. Set AOUCOV_REPO to override.
REPO_DIR = os.environ.get("AOUCOV_REPO") or next(
    (p for p in (os.path.expanduser("~/repos/aou-covariance"),
                 os.path.expanduser("~/repos/AOU-covariance"),
                 os.path.expanduser("~/aou-covariance"))
     if os.path.isdir(os.path.join(p, "src"))),
    None,
)
assert REPO_DIR and os.path.isdir(os.path.join(REPO_DIR, "src")), (
    f"cannot find the repo checkout (REPO_DIR={REPO_DIR!r}) — "
    f"set AOUCOV_REPO to the checkout path"
)

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
WS_GS       = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231/phenotypic_covariance_v9"
PROJECT_DIR = "phenotypic_covariance_v9"
RUN_DIR     = "1kg_eur"

R_DIR = f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/{RUN_DIR}"
R_GS  = f"{WS_GS}/{RUN_DIR}"

SHARDS_DIR  = f"{R_DIR}/03_grm/shards"                    # 06_grm_shards.ipynb
PHENO_DIR   = f"{R_DIR}/05_phenotypes/residualized"       # 05_phenotype_residualize.ipynb
CLASSES_GS  = f"{R_GS}/04_relatedness/king_classification/deg1_classified.tsv"
CROSSPRODUCTS_GS = f"{R_GS}/06_crossproducts"

# 09_estimators.ipynb globs "{WORK_DIR}/*_merged.full.tsv" with
# WORK_DIR = ~/grm_pheno_cov_{SAMPLE_SET}, so it needs SAMPLE_SET = "1kg_eur"
# to pick these up.
WORK_DIR = os.path.expanduser(f"~/grm_pheno_cov_{RUN_DIR}")
os.makedirs(WORK_DIR, exist_ok=True)

NBLOCKS = 50          # delete-block jackknife blocks
SEED    = 1
N_CONCURRENT = 8      # see the machine note above

# How many phenotypes to batch into a single pass over a shard. Higher = fewer
# shard traversals but more resident memory; the cell below prints the estimate.
# This is the knob to turn if the machine is smaller or the phenotype count grows.
PHENO_BATCH = 96

print(f"repo   : {REPO_DIR}")
print(f"shards : {SHARDS_DIR}")
print(f"pheno  : {PHENO_DIR}")
print(f"classes: {CLASSES_GS}")
print(f"work   : {WORK_DIR}")
print(f"bucket : {CROSSPRODUCTS_GS}")

## Build the tool, generate the bins

`src/make_bins.py --wide` gives 153 bins instead of the default 236, doubling
the mid and coarse widths. Its own docstring says why that matters *here
specifically*: the first-degree pairs number in the thousands and spread over
~30 bins at width 0.01, so splitting them into PO and FS leaves very few pairs
per bin. Wide bins keep each class's bins populated enough to mean anything.

In [ ]:
TOOL_DIR = f"{REPO_DIR}/src"
TOOL_BIN = f"{TOOL_DIR}/grm_class_tool"
BINS_FILE = f"{WORK_DIR}/bins_wide.txt"

subprocess.run(["make"], cwd=TOOL_DIR, check=True)
assert os.path.isfile(TOOL_BIN), f"missing tool binary: {TOOL_BIN}"

subprocess.run(["python3", f"{TOOL_DIR}/make_bins.py", "--out", BINS_FILE, "--wide"],
               check=True)
N_BINS = sum(1 for line in open(BINS_FILE) if line.strip())
print(f"tool: {TOOL_BIN}")
print(f"bins: {BINS_FILE}  ({N_BINS} bins)")

## Discover the shards

Discovered by glob, with the `--parallel` index parsed from the `.grm.bin.<k>`
suffix plink appends, rather than assuming the filename prefix — that prefix is
whatever `--out` the shard job used and has differed between runs.

`N_SHARDS` is then *derived* from what was found rather than hardcoded, and
checked two ways: the indices must be exactly `1..N`, and the shard bytes must
account for exactly `N_IDS*(N_IDS+1)/2` float32 entries. `--parallel k N` row
ranges depend on `N`, so a wrong value silently maps the wrong bytes to the
wrong pairs; the byte reconciliation is what makes that impossible to miss.

In [ ]:
shard_bins = {}
for path in sorted(glob.glob(f"{SHARDS_DIR}/*.grm.bin*")):
    m = re.search(r"\.grm\.bin\.(\d+)$", path)
    assert m, f"cannot parse a --parallel index from {path!r}"
    k = int(m.group(1))
    assert k not in shard_bins, f"two files claim shard {k}"
    shard_bins[k] = path

N_SHARDS = len(shard_bins)
assert N_SHARDS and sorted(shard_bins) == list(range(1, N_SHARDS + 1)), (
    f"shard indices are {sorted(shard_bins)} — expected a complete 1..N run; "
    f"did 06_grm_shards.ipynb finish?"
)

id_paths = sorted(glob.glob(f"{SHARDS_DIR}/*.grm.id"))
assert id_paths, f"no .grm.id under {SHARDS_DIR}"
GRM_ID = id_paths[0]

grm_ids = pd.read_csv(GRM_ID, sep=r"\s+", header=None, names=["FID", "IID"], dtype=str)
N_IDS = len(grm_ids)
assert grm_ids["IID"].is_unique, "duplicate IIDs in .grm.id — the tool keys pair classes on IID"

shard_bytes = sum(os.path.getsize(p) for p in shard_bins.values())
expected_entries = N_IDS * (N_IDS + 1) // 2
assert shard_bytes == expected_entries * 4, (
    f"shards hold {shard_bytes / 4:,.0f} entries but {N_IDS:,} individuals imply "
    f"{expected_entries:,} — shards missing, truncated, or from a different run"
)

print(f"{N_SHARDS} shards, {shard_bytes / 1024**3:.1f} GiB, reconcile with .grm.id")
print(f"N_IDS = {N_IDS:,}   ({expected_entries:,} GRM entries)")
print(f"GRM_ID = {GRM_ID}")
print(f"distinct FIDs in .grm.id: {sorted(grm_ids['FID'].unique())[:5]}")

## Pair classes

`deg1_classified.tsv` is keyed on **IID alone** — plink writes `FID = 0` for
every row of a pedigree-free `.grm.id`, so FID carries no information. Checked
here against the `.grm.id` before submitting anything, because the failure mode
is silent: an ID or format mismatch maps *zero* pairs and produces a clean run
in which every classed pair falls into `other`, indistinguishable from "there
were no classed pairs" unless you look.

In [ ]:
CLASSES_FILE = f"{WORK_DIR}/deg1_classified.tsv"
subprocess.run(["gcloud", "storage", "cp", CLASSES_GS, CLASSES_FILE], check=True)

classes = pd.read_csv(CLASSES_FILE, sep="\t", dtype=str)
for col in ("IID1", "IID2", "cls"):
    assert col in classes.columns, (
        f"{CLASSES_FILE} has no {col!r} column (found {list(classes.columns)}) — "
        f"the tool defaults to IID1/IID2/cls"
    )

grm_iid_set = set(grm_ids["IID"])
in_grm = classes["IID1"].isin(grm_iid_set) & classes["IID2"].isin(grm_iid_set)
print(f"{len(classes):,} classified pairs")
print(classes["cls"].value_counts().to_string())
print(f"\nboth IIDs present in .grm.id: {int(in_grm.sum()):,} / {len(classes):,}")
assert in_grm.any(), (
    "no classified pair has both IIDs in the .grm.id — the classification and the "
    "GRM are from different sample sets, or the columns hold something other than IIDs"
)
if not in_grm.all():
    print(f"NOTE: {int((~in_grm).sum()):,} pairs reference IIDs outside the GRM; the tool "
          f"will report these as 'unmapped' and ignore them")

# Expected per-class mapped counts, to reconcile against the tool's own report later.
expected_class_pairs = classes[in_grm]["cls"].value_counts().to_dict()
N_EXPECTED_MAPPED = int(in_grm.sum())

## Phenotypes

Discovered from the filenames `05_phenotype_residualize.ipynb` writes:
`<phenotype>__<transform>__<covariate_set>.pheno`. Set `TARGET_PHENOTYPES` to a
subset to run fewer; `None` runs everything found.

In [ ]:
TARGET_PHENOTYPES = None   # e.g. {"height", "bmi"}, or None for all

combos = []
for fname in sorted(os.listdir(PHENO_DIR)):
    if not fname.endswith(".pheno"):
        continue
    stem = fname[: -len(".pheno")]
    parts = stem.rsplit("__", 2)
    if len(parts) != 3:
        continue
    phenotype_name, transform, covariate_set = parts
    if TARGET_PHENOTYPES is not None and phenotype_name not in TARGET_PHENOTYPES:
        continue
    combos.append({"tag": stem, "phenotype": phenotype_name,
                   "transform": transform, "covariate_set": covariate_set,
                   "fname": fname})

assert combos, f"no .pheno files matched in {PHENO_DIR}"
print(f"{len(combos)} combos: {len({c['phenotype'] for c in combos})} phenotypes x "
      f"{len({c['transform'] for c in combos})} transforms x "
      f"{len({c['covariate_set'] for c in combos})} covariate sets")

## FID alignment

The tool joins phenotypes on `(FID, IID)`, but `write_grm_pheno()` in
`scripts/local/residualize_lib.R` emits `FID = IID = person_id` while the
`.grm.id` has `FID = 0`. Left alone, every lookup misses, every phenotype reads
as NaN and every bin comes out empty. So each `.pheno` is rewritten with the
`FID` taken from the real `.grm.id`, keyed on `IID`, with a provenance header
recording the source, the `.grm.id` used, and a fingerprint of the ID set.

In [ ]:
id_map = dict(zip(grm_ids["IID"], grm_ids["FID"]))


def build_aligned_pheno(pheno_path, out_path):
    pheno = pd.read_csv(pheno_path, sep=r"\s+", dtype={"FID": str, "IID": str})
    n_source = len(pheno)
    pheno["FID"] = pheno["IID"].map(id_map)
    unmatched = int(pheno["FID"].isna().sum())
    if unmatched:
        pheno = pheno.dropna(subset=["FID"])
    n_matched = len(pheno)

    id_fingerprint = hashlib.sha256(
        "\n".join(f"{fid} {iid}" for fid, iid in sorted(zip(pheno["FID"], pheno["IID"])))
        .encode()
    ).hexdigest()

    with open(out_path, "w") as f:
        f.write(f"# source={pheno_path}\n")
        f.write(f"# generated={datetime.now(timezone.utc).isoformat()}\n")
        f.write(f"# grm_id={GRM_ID}\n")
        f.write(f"# n_source_rows={n_source}\n")
        f.write(f"# n_matched_grm={n_matched}\n")
        f.write(f"# n_dropped_unmatched={unmatched}\n")
        f.write(f"# id_set_sha256={id_fingerprint}\n")
    pheno.to_csv(out_path, sep=" ", index=False, columns=["FID", "IID", "Y"], mode="a")
    return out_path, n_matched


aligned_rows = []
for c in combos:
    c["aligned"], n_matched = build_aligned_pheno(
        f"{PHENO_DIR}/{c['fname']}", f"{WORK_DIR}/{c['tag']}_aligned.pheno"
    )
    aligned_rows.append(n_matched)

print(f"{len(combos)} phenotype files aligned")
print(f"rows matching the GRM: min {min(aligned_rows):,}, max {max(aligned_rows):,}")
assert min(aligned_rows) > 0, (
    "a phenotype matched zero GRM rows — FID/IID alignment is wrong, and every "
    "bin would come out empty"
)

## Batch the work

Tasks are `(shard, phenotype-batch)`. Each task is one traversal of one shard
that accumulates every phenotype in its batch, so the number of shard
traversals is `N_SHARDS * ceil(n_combos / PHENO_BATCH)` rather than
`N_SHARDS * n_combos`.

The memory estimate is the reason `PHENO_BATCH` exists. Per process:

- phenotype values, stored transposed: `K * N_IDS * 8` bytes
- accumulators: `n_classes * N_BINS * K * 24 * (1 + NBLOCKS)` bytes
  (`acc_full` plus the `NBLOCKS` delete-block copies)

In [ ]:
import math

batches = [combos[i:i + PHENO_BATCH] for i in range(0, len(combos), PHENO_BATCH)]
n_classes = len(expected_class_pairs) + 1        # real classes plus 'other'

K = max(len(b) for b in batches)
pheno_bytes = K * N_IDS * 8
acc_bytes = n_classes * N_BINS * K * 24 * (1 + NBLOCKS)
per_proc_gb = (pheno_bytes + acc_bytes) / 1024**3
total_gb = per_proc_gb * N_CONCURRENT

print(f"{len(combos)} combos in {len(batches)} batches of <= {PHENO_BATCH}")
print(f"{N_SHARDS} shards x {len(batches)} batches = {N_SHARDS * len(batches)} shard traversals")
print(f"  (one pass per combo would have been {N_SHARDS * len(combos):,})")
print(f"\nper process: {pheno_bytes / 1024**3:.2f} GB phenotypes + "
      f"{acc_bytes / 1024**3:.2f} GB accumulators = {per_proc_gb:.2f} GB")
print(f"at N_CONCURRENT={N_CONCURRENT}: {total_gb:.1f} GB resident")
assert total_gb < 10, (
    f"{total_gb:.1f} GB across {N_CONCURRENT} workers is too close to an "
    f"n1-highcpu-16's 14.4 GB — lower PHENO_BATCH or N_CONCURRENT"
)

## Accumulate

Each task writes one `.acc.tsv` per phenotype via `--out-pattern`, whose
`{name}` placeholder the tool substitutes with the first column of the
`--pheno-list` file. A task is skipped only if *every* accumulator it would
write already exists — a partially-written batch re-runs in full, which is
cheap relative to getting it wrong.

`accumulate`'s `[INFO]` lines on stderr are captured and parsed rather than
discarded: they carry the `read / mapped / unmapped / duplicate` pair-class
counts and the per-class binned-pair hits, which is the only way to tell a
working classification from one that silently matched nothing.

In [ ]:
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed

RE_PC = re.compile(r"pair classes: (\d+) read, (\d+) mapped, (\d+) unmapped, (\d+) duplicate")
RE_HITS = re.compile(r"\[INFO\] (\S+) by class: (.+?) -> ")


LOG_DIR = f"{WORK_DIR}/logs"
os.makedirs(LOG_DIR, exist_ok=True)


def acc_path(tag, k):
    return f"{WORK_DIR}/{tag}_shard{k}.acc.tsv"


def nonempty(path):
    return os.path.isfile(path) and os.path.getsize(path) > 0


def run_task(k, batch_idx, batch):
    """One traversal of shard k accumulating every phenotype in `batch`."""
    outputs = [acc_path(c["tag"], k) for c in batch]
    if all(nonempty(p) for p in outputs):
        return {"k": k, "batch": batch_idx, "skipped": True}

    list_path = f"{WORK_DIR}/pheno_list_b{batch_idx}.tsv"
    with open(list_path, "w") as f:
        for c in batch:
            f.write(f"{c['tag']}\t{c['aligned']}\n")

    # Stream to a log file rather than capture_output: the tool reports progress
    # every 10% on stderr, and capturing it would hide that until the process
    # exits -- which for a multi-hour task makes a working run indistinguishable
    # from a hung one. `tail -f` any of these to watch.
    log_path = f"{LOG_DIR}/shard{k}_batch{batch_idx}.log"
    with open(log_path, "w") as lf:
        proc = subprocess.run(
            [TOOL_BIN, "accumulate",
             "--grm-id", GRM_ID,
             "--shard", shard_bins[k],
             "--parallel", str(k), str(N_SHARDS),
             "--pheno-list", list_path,
             "--bins", BINS_FILE,
             "--pair-classes", CLASSES_FILE,
             "--nblocks", str(NBLOCKS), "--seed", str(SEED),
             "--out-pattern", WORK_DIR + "/{name}_shard" + str(k) + ".acc.tsv"],
            stdout=lf, stderr=lf, text=True,
        )
    stderr_text = open(log_path).read()
    if proc.returncode != 0:
        raise RuntimeError(
            f"accumulate failed on shard {k}, batch {batch_idx} "
            f"(full log: {log_path})\n{stderr_text[-4000:]}"
        )
    missing = [p for p in outputs if not nonempty(p)]
    if missing:
        raise RuntimeError(
            f"shard {k} batch {batch_idx} exited 0 but wrote no output for "
            f"{len(missing)} phenotypes, e.g. {missing[0]}"
        )

    pc = RE_PC.search(stderr_text)
    hits = defaultdict(int)
    for _name, body in RE_HITS.findall(stderr_text):
        for token in body.split():
            cls, _, n = token.partition("=")
            if n:
                hits[cls] += int(n)
    return {"k": k, "batch": batch_idx, "skipped": False,
            "pc": tuple(int(x) for x in pc.groups()) if pc else None,
            "hits": dict(hits)}


tasks = [(k, bi, b) for k in sorted(shard_bins) for bi, b in enumerate(batches)]
print(f"{len(tasks)} tasks, N_CONCURRENT={N_CONCURRENT}")
print(f"expecting {len(combos) * N_SHARDS:,} accumulator files in {WORK_DIR}")
print(f"progress logs: {LOG_DIR}/shard<k>_batch<b>.log  (tail -f one to watch)")
print("\nThis cell is the long one. To check it from a terminal:")
print(f"  ls {WORK_DIR}/*.acc.tsv | wc -l          # climbs to {len(combos) * N_SHARDS:,}")
print(f"  tail -f {LOG_DIR}/shard1_batch0.log      # per-shard percentage")

results, done = [], 0
with ProcessPoolExecutor(max_workers=N_CONCURRENT) as ex:
    futures = {ex.submit(run_task, *t): t for t in tasks}
    for fut in as_completed(futures):
        results.append(fut.result())     # re-raises with the tool's stderr attached
        done += 1
        print(f"  [{done}/{len(tasks)}] shard {results[-1]['k']} batch "
              f"{results[-1]['batch']}{' (skipped)' if results[-1]['skipped'] else ''}")

### Reconcile the pair classes

Per `src/README.md`: summing the per-class hits across shards should reconcile
against the pair counts in the classification file, less pairs dropped for a
missing phenotype or for falling outside every bin. Each shard reads the whole
classification file, so `mapped` is per-shard and should be identical
everywhere; the *hits* are the pairs actually binned, and those partition
across shards by row range.

A zero here is the failure this check exists for. Hits can legitimately fall
slightly short of the mapped count — a pair is only binned if both members have
a phenotype value and `a_ij` lands inside the bin range — but they should not be
wildly short, and never zero.

In [ ]:
ran = [r for r in results if not r["skipped"]]
if not ran:
    print("every task was skipped (outputs already present) — nothing to reconcile")
else:
    mapped = {r["pc"][1] for r in ran if r["pc"]}
    assert mapped, "no task reported a 'pair classes:' line — was --pair-classes passed?"
    assert len(mapped) == 1, f"shards disagree on the mapped pair count: {mapped}"
    n_mapped = mapped.pop()
    print(f"pair classes mapped per shard: {n_mapped:,} "
          f"(expected {N_EXPECTED_MAPPED:,} from the local check)")
    assert n_mapped > 0, (
        "the tool mapped ZERO pairs -- every classed pair fell into 'other'. Check that "
        "deg1_classified.tsv holds IIDs matching the .grm.id."
    )
    if n_mapped != N_EXPECTED_MAPPED:
        print("NOTE: differs from the local count; the tool also drops self-pairs and "
              "duplicate orientations, which the local check does not")
    # totals[cls] sums hits over every (shard, phenotype) instance. A full run has
    # n_shards * n_combos instances and each phenotype is binned once per shard, so
    # the whole-GRM count for one phenotype is totals * n_shards / n_instances.
    totals = defaultdict(int)
    for r in ran:
        for cls, n in r["hits"].items():
            totals[cls] += n
    n_instances = sum(len(batches[r["batch"]]) for r in ran)
    n_shards_ran = len({r["k"] for r in ran})
    scale = n_shards_ran / n_instances

    print("\nbinned pairs per phenotype, over the whole GRM:")
    for cls in sorted(totals, key=lambda c: -totals[c]):
        per_pheno = totals[cls] * scale
        exp = expected_class_pairs.get(cls)
        exp_s = f"   ({exp:,} pairs in the classification file)" if exp else ""
        print(f"  {cls:>8}: {per_pheno:15,.0f}{exp_s}")
        if cls != "other":
            assert per_pheno > 0, (
                f"class {cls} binned zero pairs despite being mapped — check that the "
                f"phenotypes cover these individuals and a_ij falls inside the bin range"
            )
    print("\nclassification is live: every named class binned a nonzero number of pairs")


## Merge and upload

`merge` combines a combo's per-shard accumulators into `<tag>_merged.full.tsv`
(one row per `(class, bin)`) plus `<tag>_merged.jk.tsv` (the delete-block
values). The `pooled` class is synthesised at merge time as the sum over the
real classes, so an unclassed comparison needs no separate run.

Merge and upload are checked independently: a local `.full.tsv` existing does
not mean it ever reached the bucket, e.g. if a run was interrupted between the
two steps.

In [ ]:
def list_bucket_files(prefix):
    r = subprocess.run(["gcloud", "storage", "ls", prefix], capture_output=True, text=True)
    if r.returncode != 0:
        return set()
    return {ln.strip().rsplit("/", 1)[-1] for ln in r.stdout.splitlines() if ln.strip()}


bucket_files = list_bucket_files(f"{CROSSPRODUCTS_GS}/*")
print(f"{len(bucket_files)} files already in {CROSSPRODUCTS_GS}")


def merge_and_upload(tag):
    acc_files = [acc_path(tag, k) for k in sorted(shard_bins)]
    missing = [p for p in acc_files if not nonempty(p)]
    assert not missing, f"{tag}: {len(missing)} accumulators missing, e.g. {missing[0]}"

    acc_list = f"{WORK_DIR}/{tag}_acc_list.txt"
    with open(acc_list, "w") as f:
        f.write("\n".join(acc_files) + "\n")

    out_prefix = f"{WORK_DIR}/{tag}_merged"
    full_tsv = f"{out_prefix}.full.tsv"
    if not nonempty(full_tsv):
        subprocess.run([TOOL_BIN, "merge", "--acc-list", acc_list, "--bins", BINS_FILE,
                        "--nblocks", str(NBLOCKS), "--out-prefix", out_prefix], check=True)
    if os.path.basename(full_tsv) not in bucket_files:
        subprocess.run(["gcloud", "storage", "cp", full_tsv, f"{CROSSPRODUCTS_GS}/"],
                       check=True)
    return full_tsv


merged = []
with ProcessPoolExecutor(max_workers=N_CONCURRENT) as ex:
    futures = {ex.submit(merge_and_upload, c["tag"]): c["tag"] for c in combos}
    for fut in as_completed(futures):
        merged.append(fut.result())
        if len(merged) % 20 == 0 or len(merged) == len(combos):
            print(f"  {len(merged)}/{len(combos)} combos merged + uploaded")

print(f"\n{len(merged)}/{len(combos)} merged -> {CROSSPRODUCTS_GS}")

### Check one merged result

The classes present, and whether the first-degree bins actually hold pairs in
the PO and FS classes separately — the point of the whole exercise. `pooled`
should equal the sum of the real classes by construction.

In [ ]:
sample = pd.read_csv(sorted(merged)[0], sep="\t")
print(f"{os.path.basename(sorted(merged)[0])}: {len(sample)} rows")
print(f"classes: {sorted(sample['class'].unique())}\n")

deg1 = sample[(sample["bin_midpoint"] > 0.35) & (sample["bin_midpoint"] < 0.7)]
print("first-degree bins (0.35 < a_ij < 0.7), pairs per class:")
print(deg1.groupby("class")["full_n"].sum().sort_values(ascending=False).to_string())

pooled = sample[sample["class"] == "pooled"]["full_n"].sum()
real = sample[sample["class"] != "pooled"]["full_n"].sum()
print(f"\npooled n = {pooled:,.0f}, sum over real classes = {real:,.0f}",
      "(match)" if pooled == real else "(MISMATCH — investigate)")

## Copy notebook to bucket

In [ ]:
_nb = os.path.join(REPO_DIR, "notebooks/08_accumulate.ipynb")
_gs = f"{R_GS}/notebooks/08_accumulate.ipynb"
subprocess.run(["gcloud", "storage", "cp", _nb, _gs], check=True)
print(f"notebook -> {_gs}")